# La curva de precios en producción · TFM Energía UCM

Cómo se genera y se republica la curva de 20 años, y qué hay que tener al día para que sirva.

## La cadena

```
  crons de ingesta          llenan la base: precio, generación, ECMWF, commodities
          ↓
  construir_matriz_produccion.py     reconstruye matriz_produccion desde la base
          ↓
    ┌─────┴─────┐
    ↓           ↓
 predicción   generar_curva.py       publica el .npy y registra en app_curve_run
   de D+1
```

**La matriz va primero, y siempre.** La curva saca de ella dos cosas que caducan:

- **Las anclas.** El gas, la demanda y la capacidad del escenario salen del **último año
  observado**. Con la matriz parada se quedan en la fecha en que se paró, y la curva envejece
  sin que nada avise: sigue generándose, sigue teniendo buena pinta, y cada día se parece
  menos a la realidad.
- **El arranque.** El tramo simulado empieza en `último_día + 1`. Con la matriz de ayer, la
  curva no cubre mañana, y un caso que pida mañana se queda sin precio.

## Y una decisión de diseño

La curva se publica como **artefacto**: un `.npy` con los escenarios, un índice y un
`meta.json`. Quien la consulta —o quien optimiza una batería sobre ella— **no necesita la
matriz, ni la base, ni los modelos**. Solo el fichero.

Antes, optimizar obligaba a reconstruir la curva entera: leer la matriz, ajustar rendimientos,
ajustar la curva de oferta y sortear 50 escenarios de 178.000 horas. Minutos, y solo para
quien tuviera montada toda la cadena.

In [ ]:
import sys, subprocess, time, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
sys.path.append(str(REPO / "scripts"))
sys.path.append(str(REPO / "production" / "curva"))
PY = sys.executable

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"repo: {REPO}")

## 1 · Qué hay ahora mismo

Antes de tocar nada: en qué estado están la matriz y la curva, y si van al día.

In [ ]:
import curva_fundamental as cfun
from generar_curva import SALIDA
importlib = __import__("importlib"); importlib.reload(cfun)

hoy = pd.Timestamp.today().normalize()

# ── la matriz ──────────────────────────────────────────────────────────────
mm = cfun.meta_matriz("produccion")
if mm:
    gen = pd.Timestamp(str(mm["generada"])[:19])
    print(f"  MATRIZ produccion")
    print(f"    hash        {mm['hash']}")
    print(f"    generada    {gen:%Y-%m-%d %H:%M}  ({(hoy-gen.normalize()).days} días)")
    print(f"    llega hasta {mm.get('modelo_end', '?')}")
    print(f"    {mm['filas']:,} filas x {mm['columnas']} columnas")
else:
    print("  MATRIZ: no hay meta.json — ¿se ha construido alguna vez?")

# ── la curva ───────────────────────────────────────────────────────────────
f = SALIDA / "curva_meta.json"
if f.exists():
    cm = json.loads(f.read_text(encoding="utf-8"))
    pub = pd.Timestamp(cm["generado"][:19])
    print(f"\n  CURVA publicada")
    print(f"    generada    {pub:%Y-%m-%d %H:%M}  ({(hoy-pub.normalize()).days} días)")
    print(f"    matriz      {cm.get('matriz','?')} · hash {cm.get('matriz_hash','?')}")
    print(f"    cubre       {cm['desde']} -> {cm['hasta']}")
    print(f"    {cm['escenarios']} escenarios x {cm['horas']:,} horas")
    # Lo que importa NO es si la curva llega a mañana, sino si hay HUECO entre el último
    # dato observado y donde empieza la curva. Si el precio de mañana ya está publicado,
    # la curva empieza pasado mañana y eso es correcto: comparar contra "mañana" mandaría
    # a republicar todos los días sin motivo.
    ult = pd.Timestamp(cm["ultimo_dato_observado"])
    hueco = (pd.Timestamp(cm["desde"]) - ult).days - 1
    print(f"\n    dato real hasta {ult:%Y-%m-%d}, curva desde {cm['desde']}")
    print(f"    hueco de {hueco} días  "
          + ("(ninguno: la curva empalma con el dato real)" if hueco == 0
             else "-> HAY DÍAS SIN PRECIO, republica"))
    if cm.get("matriz_hash") != mm.get("hash"):
        print(f"    AVISO: la curva se generó con otra matriz "
              f"({cm.get('matriz_hash')} vs {mm.get('hash')}). Republícala.")
else:
    print("\n  CURVA: no hay ninguna publicada")

## 2 · Reconstruir la matriz

**Tarda unos minutos** — lee unas quince tablas de la base — así que está apagado por defecto.
Enciéndelo solo si el diagnóstico de arriba dice que la matriz está atrasada.

La reconstrucción activa cuatro palancas del constructor del equipo, todas con el valor por
defecto que no cambia nada para nadie más:

| palanca | qué hace |
|---|---|
| `MODELO_END` | mueve el último día que se predice |
| `EXIGIR_TARGET` | permite filas sin precio publicado |
| `ERA5_PREFERIR_ECMWF` | los lags meteo salen de la previsión, no del reanálisis |
| `ESPINA_CALENDARIO` | la fila de mañana no depende de que haya precio |

La última es la que permite predecir mañana entre las 11:00 y las 13:00, que es la ventana en
que el mercado aún no ha casado.

In [ ]:
RECONSTRUIR = False          # ponlo en True si la matriz está atrasada

manana = (hoy + pd.Timedelta(days=1)).date()
if RECONSTRUIR:
    t0 = time.time()
    r = subprocess.run([PY, "-u", "scripts/construir_matriz_produccion.py",
                        "--hasta", str(manana)],
                       cwd=REPO, capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    print(r.stdout[-3000:])
    if r.returncode:
        print(r.stderr[-2000:])
        raise RuntimeError("la matriz ha fallado: NO republiques la curva sobre la vieja")
    print(f"\n  {time.time()-t0:.0f}s")
    importlib.reload(cfun)
else:
    print(f"  desactivado. La matriz llega a {cfun.meta_matriz('produccion').get('modelo_end','?')}")
    print(f"  y mañana es {manana}.")

## 3 · Lo que la curva aprende de la matriz

Antes de generarla, ver qué sale del panel. Son los cinco pasos del ajuste, y ninguno
depende de los modelos de D+1: la curva es un objeto independiente.

In [ ]:
P = cfun.panel("produccion")
potencial, ir = cfun.rendimientos(P)
D = cfun.con_residual(P, potencial)
precio_of, ic = cfun.curva_oferta(D)

print(f"  panel          {len(P):,} horas · {P.ano.min()}-{P.ano.max()}")
print(f"  rendimiento    η solar {ir['eta_solar']} (R² {ir['R2_solar']}) · "
      f"η eólica {ir['eta_eolica']} (R² {ir['R2_eolica']})")
print(f"  curva oferta   ajustada {ic['ajustada_desde']}-{ic['hasta']} · "
      f"{ic['bins']} tramos")
print(f"                 k de {ic['k_min']} a {ic['k_max']} · "
      f"β del gas {ic['beta_gas']}")
print(f"                 P(precio≤0) de {ic['p0_max']} a {ic['p0_min']}")

# las anclas, que es lo que caduca con la matriz
obs = P[P.ano == P.ano.max()]
print(f"\n  ANCLAS del año en curso ({P.ano.max()}), que es lo que envejece:")
print(f"    gas       {obs.gas_mibgas.mean():8.1f} €/MWh")
print(f"    demanda   {obs.demanda.mean():8,.0f} MW")
print(f"    solar     {obs.solar_gw.mean():8.1f} GW")
print(f"    eólica    {obs.eolica_gw.mean():8.1f} GW")

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(ic["centro"], ic["k"], "o-", ms=3, color="#c0392b", lw=2.2,
        label="k = precio / gas^β")
ax.set_xlabel("demanda residual (MW)"); ax.set_ylabel("k", color="#c0392b")
ax.tick_params(axis="y", labelcolor="#c0392b")
b = ax.twinx(); b.grid(False)
b.plot(ic["centro"], ic["p0"], "s--", ms=3, color="#2874a6", lw=1.8)
b.fill_between(ic["centro"], 0, ic["p0"], alpha=.12, color="#2874a6")
b.set_ylabel("P(precio ≤ 0)", color="#2874a6"); b.tick_params(axis="y", labelcolor="#2874a6")
ax.set_title("La curva de oferta que se acaba de ajustar", fontsize=12)
plt.tight_layout(); plt.show()

## 4 · Publicar la curva

Esto sí es rápido: el ajuste ya está hecho arriba, y sortear los escenarios cuesta segundos.

Se escriben tres ficheros y, con `registrar`, dos tablas:

```
curva_escenarios.npy     (escenarios, horas) float32 — 33 MB con 50 escenarios
curva_indice.parquet     el eje temporal
curva_meta.json          qué escenarios de gas, demanda y capacidad se usaron

app_curve_run            la cabecera, con el hash de la matriz
app_curve_hourly         los percentiles horarios, que es lo que consulta la web
```

**Los 50 escenarios no van a Postgres.** Son 8,8 millones de flotantes; se quedan en el `.npy`
y `artifact_path` apunta a él.

In [ ]:
ESCENARIOS = 50
REGISTRAR = True             # escribir también en app_curve_run y app_curve_hourly

from generar_curva import publicar, registrar as registrar_curva

t0 = time.time()
meta, sims, idx = publicar(hasta=2046, n=ESCENARIOS, matriz="produccion")
print(f"\n  {time.time()-t0:.0f}s")

if REGISTRAR:
    try:
        cid = registrar_curva(meta, sims, idx)
    except Exception as e:
        print(f"  no se ha podido registrar: {type(e).__name__}: {str(e)[:120]}")
        cid = None
else:
    cid = None

## 5 · Verla

Desde el artefacto y nada más. Es el mismo código que usa
[`production/curva/ver_curva.py`](../production/curva/ver_curva.py).

In [ ]:
from ver_curva import cargar, dibujar
px, dias, cm = cargar()
anos = dias.year.to_numpy()

print(f"  {cm['escenarios']} escenarios · {cm['desde']} -> {cm['hasta']}")
print(f"  media {px.mean():.2f} €/MWh · horas ≤ 0: {(px<=0).mean():.1%}\n")
print(f"  {'año':>5s} {'P10':>8s} {'P50':>8s} {'P90':>8s} {'h≤0':>7s}")
print("  " + "-"*42)
for y in np.unique(anos):
    m = anos == y
    if m.sum() < 365:
        continue
    q = np.percentile(px[:, m, :], [10, 50, 90])
    print(f"  {y:5d} {q[0]:8.1f} {q[1]:8.1f} {q[2]:8.1f} "
          f"{(px[:,m,:]<=0).mean()*100:6.1f}%")

dibujar(px, dias, cm, mes=f"{(dias[0].year + dias[-1].year)//2}-07")

## 6 · El cron

Todo lo anterior, sin intervención, está en
[`production/curva/cron_diario.py`](../production/curva/cron_diario.py): reconstruye la matriz
y republica la curva, en ese orden.

```
30 3 * * *  /home/ubuntu/tfm-env/bin/python -u \
            /home/ubuntu/scripts/production/curva/cron_diario.py \
            >> /home/ubuntu/scripts/logs/cron_curva.log 2>&1
```

**A las 03:30** por dos motivos: los crons de ingesta ya han traído el día anterior, y la
predicción de las 11:00 usa la misma matriz, así que tiene que estar lista antes.

**El `-u` no es opcional.** Sin él, Python acumula la salida al redirigir a fichero y el log
parece vacío durante minutos aunque el proceso esté trabajando — cuesta media hora de
depuración averiguar que no estaba colgado.

### Lo que hace si algo falla

**Si la matriz falla, la curva NO se republica.** Es deliberado: más vale una curva de ayer,
con su fecha bien puesta, que una de hoy construida sobre datos de la semana pasada. La
segunda miente y no lo parece — se genera igual, tiene buena pinta, y sus anclas son de otra
época.

### Cada cuánto

A diario, y no porque las anclas se muevan tanto — son medias del año en curso y apenas
cambian en 24 horas. Es por el **arranque**: el tramo simulado empieza en `último_día + 1`, así
que la curva tiene que republicarse para cubrir mañana.

In [ ]:
# comprobación de coherencia: ¿la curva publicada casa con la matriz de ahora?
mm = cfun.meta_matriz("produccion")
cm = json.loads((SALIDA / "curva_meta.json").read_text(encoding="utf-8"))

print(f"  {'':16s} {'matriz':>12s}   {'curva':>12s}")
print("  " + "-"*46)
print(f"  {'hash':16s} {str(mm.get('hash')):>12s}   {str(cm.get('matriz_hash')):>12s}")
print(f"  {'último día':16s} {str(mm.get('modelo_end')):>12s}   "
      f"{str(cm.get('ultimo_dato_observado')):>12s}")
igual = mm.get("hash") == cm.get("matriz_hash")
print(f"\n  {'coherentes' if igual else 'DESALINEADAS: republica la curva'}")
# El HUECO, no la fecha de mañana: si el precio de mañana ya está publicado, la curva
# empieza pasado mañana y eso es lo correcto.
ult = pd.Timestamp(cm["ultimo_dato_observado"])
hueco = (pd.Timestamp(cm["desde"]) - ult).days - 1
print(f"  dato real hasta {ult:%Y-%m-%d} · curva desde {cm['desde']} · hueco {hueco} días")
print(f"  {'empalma: ningún día sin precio' if hueco == 0 else 'HAY HUECO: republica'}")

# y si la matriz ha avanzado desde que se publicó la curva, republica igualmente
if str(mm.get("modelo_end", "")) > str(cm.get("ultimo_dato_observado", "")):
    print(f"  la matriz ya llega a {mm['modelo_end']}: republica para aprovecharlo")